In [11]:
!pip install sentence-transformers rank_bm25 langchain-text-splitters langchain-community pypdf transformers -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [12]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

from sentence_transformers import SentenceTransformer, CrossEncoder
print("sentence_transformers ✅")

from rank_bm25 import BM25Okapi
print("rank_bm25 ✅")

from langchain_text_splitters import RecursiveCharacterTextSplitter
print("langchain_text_splitters ✅")

from langchain_community.document_loaders import PyPDFLoader
print("PyPDFLoader ✅")

from transformers import T5ForConditionalGeneration, AutoTokenizer
print("transformers ✅")

PyTorch: 2.10.0+cu128
CUDA: True
sentence_transformers ✅
rank_bm25 ✅
langchain_text_splitters ✅
PyPDFLoader ✅
transformers ✅


In [13]:
!wget -q https://arxiv.org/pdf/1706.03762 -O attention.pdf
!wget -q https://arxiv.org/pdf/1810.04805 -O bert.pdf
!wget -q https://arxiv.org/pdf/2005.11401 -O rag.pdf
!wget -q https://arxiv.org/pdf/2005.14165 -O gpt3.pdf
!wget -q https://arxiv.org/pdf/2307.09288 -O llama2.pdf
!ls -la *.pdf

-rw-r--r-- 1 root root  2215244 Apr 12  2024 attention.pdf
-rw-r--r-- 1 root root   775166 Jan 22  2023 bert.pdf
-rw-r--r-- 1 root root  6768044 Jan 23  2023 gpt3.pdf
-rw-r--r-- 1 root root 13661300 Jul 22  2023 llama2.pdf
-rw-r--r-- 1 root root   885323 Jan 23  2023 rag.pdf


In [14]:
from langchain_community.document_loaders import PyPDFLoader

paths = ["attention.pdf", "bert.pdf", "rag.pdf", "gpt3.pdf", "llama2.pdf"]
docs = []
for p in paths:
    pages = PyPDFLoader(p).load()
    text = "\n".join([page.page_content for page in pages])
    docs.append(text)
    print(f"✅ {p}: {len(text)} chars, {len(pages)} pages")

✅ attention.pdf: 39629 chars, 15 pages
✅ bert.pdf: 64138 chars, 16 pages
✅ rag.pdf: 69074 chars, 19 pages
✅ gpt3.pdf: 236741 chars, 75 pages
✅ llama2.pdf: 260529 chars, 77 pages


In [15]:
import numpy as np
from collections import defaultdict
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch

class AdvancedRAG:
    def __init__(self, docs, embedding_model="BAAI/bge-small-en-v1.5",
                 cross_encoder_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
                 gen_model="google/flan-t5-large",
                 chunk_size=512, chunk_overlap=50):
        """
        docs: List[str], 每个元素是一篇文档的全文
        """
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", "。", ". ", " "],
        )

        full = "\n\n".join(docs)
        self.chunks = [c.page_content for c in splitter.create_documents([full])]
        print(f"✅ Chunking finished: {len(self.chunks)} chunks (size={chunk_size}, overlap={chunk_overlap})")

        # ---- Dense Index (bi-encoder) ----
        self.bi_encoder = SentenceTransformer(embedding_model)
        self.doc_embs = self.bi_encoder.encode(
            self.chunks, normalize_embeddings=True, show_progress_bar=True,
        )
        print(f"✅ Dense index finished: {self.doc_embs.shape}")

        # ---- Sparse Index (BM25) ----
        tokenized = [c.lower().split() for c in self.chunks]
        self.bm25 = BM25Okapi(tokenized)
        print(f"✅ BM25 index finished")

        # ---- Cross-encoder (精排) ----
        self.cross_encoder = CrossEncoder(cross_encoder_model)
        print(f"✅ Cross-encoder finished loading")

        # ---- Generator (flan-t5) ----
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.gen_tokenizer = AutoTokenizer.from_pretrained(gen_model)
        self.gen_model = T5ForConditionalGeneration.from_pretrained(gen_model).to(self.device)
        print(f"Generator finished loading: {gen_model} on {self.device}")

    # ========== 检索方法 ==========
    def _dense(self, query, k):
        q = self.bi_encoder.encode([query], normalize_embeddings=True).reshape(1, -1)
        scores = np.dot(self.doc_embs, q.T).flatten()
        idx = np.argsort(scores)[::-1][:k]
        return [(int(i), float(scores[i])) for i in idx]

    def _sparse(self, query, k):
        scores = self.bm25.get_scores(query.lower().split())
        idx = np.argsort(scores)[::-1][:k]
        return [(int(i), float(scores[i])) for i in idx]

    def _rrf(self, rankings_list, k=60, top=30):
        scores = defaultdict(float)
        for rankings in rankings_list:
            for rank, (idx, _) in enumerate(rankings):
                scores[idx] += 1 /(k+rank)
        return sorted(scores.items(), key=lambda x: -x[1])[:top]

    def retrieve(self, query, method="hybrid", top_k=3, rerank=True, retrieve_k=30):
        """
        method: "dense" | "bm25" | "hybrid"
        rerank: cross-encoder
        retrieve_k: retrieve count for dense, bm25 or hybrid
        return: List[dict] with keys: text, score, index
        """
        if method == "hybrid":
            dense = self._dense(query, retrieve_k)
            sparse = self._sparse(query, retrieve_k)
            candidates = self._rrf([dense, sparse], top=retrieve_k)
        elif method == "dense":
            candidates = self._dense(query, retrieve_k)
        elif method == "bm25":
            candidates = self._sparse(query, retrieve_k)
        else:
            raise ValueError(f"Unknown method: {method}")

        if rerank and len(candidates) > 0:
            pairs = [(query, self.chunks[i]) for i, _ in candidates]
            scores = self.cross_encoder.predict(pairs)
            reranked = sorted(
                zip([i for i, _ in candidates], scores), key=lambda x: -x[1]
            )
            final = reranked[:top_k]
        else:
            final = candidates[:top_k]

        return [
            {"text": self.chunks[i], "score": float(s), "index": i} for i, s in final
        ]

    def generate(self, query, contexts):
        context_str = "\n\n".join(c["text"] for c in contexts)
        prompt = (
            "Based on the context below, answer the question.\n\n"
            f"Context:\n{context_str}\n\n"
            f"Question: {query}\n\n"
            "Answer:"
        )

        tokens = rag.gen_tokenizer(prompt)
        print(f"Prompt length: {len(prompt)} chars, {len(tokens['input_ids'])} tokens")

        inputs = self.gen_tokenizer(
            prompt, return_tensors="pt",
            max_length=1024, truncation=True
        ).to(self.device)

        with torch.no_grad():
            outputs = self.gen_model.generate(
                **inputs,
                max_new_tokens=256,
                num_beams=4,
                early_stopping=True
            )
        return self.gen_tokenizer.decode(outputs[0], skip_special_tokens=True)

    def query(self, question, method="hybrid", top_k=3, rerank=True):
        contexts = self.retrieve(question, method=method, top_k=top_k, rerank=rerank)
        answer = self.generate(question, contexts)
        return {"answer": answer, "contexts": contexts}


rag = AdvancedRAG(docs, chunk_size=512, chunk_overlap=50)

contexts = rag.retrieve("How does attention mechanism work?")
print(f"Retrieved {len(contexts)} chunks:")
for i, c in enumerate(contexts):
    print(f"\n--- Chunk {i+1} (score: {c['score']:.3f}) ---")
    print(c["text"][:300])

result = rag.query("How does attention mechanism work?")
print("Answer:", result["answer"])



✅ Chunking finished: 1462 chunks (size=512, overlap=50)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator finished loading: google/flan-t5-large on cuda
Retrieved 3 chunks:

--- Chunk 1 (score: 7.069) ---
Attention mechanisms have become an integral part of compelling sequence modeling and transduc-
tion models in various tasks, allowing modeling of dependencies without regard to their distance in
the input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanisms
a

--- Chunk 2 (score: 4.866) ---
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence.

--- Chunk 3 (score: 3.135) ---
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all posit

In [2]:
!pip install rouge-score bert-score evaluate sacrebleu nltk -q


import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
import evaluate
print("evaluate imported ✅")
from bert_score import score as bert_score_fn
print("bert_score imported ✅")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.5 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128
CUDA: True
evaluate imported ✅
bert_score imported ✅


In [3]:
# 模拟 RAG 场景的输入输出
examples = [
    {
        "question": "What is the attention mechanism in transformers?",
        "reference": "The attention mechanism allows the model to focus on relevant parts of the input sequence when producing each output element. It computes weighted sums of value vectors, where weights are determined by the compatibility between query and key vectors.",
        "generated": "Attention mechanism lets the model attend to different parts of the input. It uses query, key, and value vectors to compute weighted combinations, allowing the model to focus on relevant information.",
        "contexts": [
            "The attention mechanism computes a weighted sum of values based on query-key compatibility.",
            "Transformers use multi-head attention to attend to information from different representation subspaces.",
        ],
    },
    {
        "question": "How does BERT pre-training work?",
        "reference": "BERT is pre-trained using two objectives: Masked Language Modeling (MLM) where 15% of tokens are masked and predicted, and Next Sentence Prediction (NSP) where the model predicts if two sentences are consecutive.",
        "generated": "BERT uses masked language modeling where random tokens are hidden and the model learns to predict them. It also uses next sentence prediction to understand sentence relationships. BERT was developed by Facebook AI Research.",
        "contexts": [
            "BERT pre-training uses MLM (masking 15% of tokens) and NSP objectives.",
            "The masked language model randomly masks tokens and trains the model to predict them from context.",
        ],
    },
]

references = [ex["reference"] for ex in examples]
candidates = [ex["generated"] for ex in examples]

In [4]:
print(references)

['The attention mechanism allows the model to focus on relevant parts of the input sequence when producing each output element. It computes weighted sums of value vectors, where weights are determined by the compatibility between query and key vectors.', 'BERT is pre-trained using two objectives: Masked Language Modeling (MLM) where 15% of tokens are masked and predicted, and Next Sentence Prediction (NSP) where the model predicts if two sentences are consecutive.']


In [5]:
print(candidates)

['Attention mechanism lets the model attend to different parts of the input. It uses query, key, and value vectors to compute weighted combinations, allowing the model to focus on relevant information.', 'BERT uses masked language modeling where random tokens are hidden and the model learns to predict them. It also uses next sentence prediction to understand sentence relationships. BERT was developed by Facebook AI Research.']


In [6]:
import evaluate

bleu = evaluate.load("sacrebleu")

for i, (ref, cand) in enumerate(zip(references, candidates)):
    result = bleu.compute(predictions=[cand], references=[[ref]])
    print(f"Example {i+1} BLEU: {result['score']:.2f}")
    # 注意 references 是 [[ref]]（list of list），因为一个 candidate 可以有多个参考答案

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Example 1 BLEU: 20.93
Example 2 BLEU: 3.18


In [7]:
rouge = evaluate.load("rouge")

results = rouge.compute(predictions=candidates, references=references)
print("ROUGE scores:")
for key, value in results.items():
    print(f"  {key}: {value:.4f}")

ROUGE scores:
  rouge1: 0.4797
  rouge2: 0.2394
  rougeL: 0.3499
  rougeLsum: 0.3499


In [8]:
from bert_score import score as bert_score_fn

P, R, F1 = bert_score_fn(
    candidates, references,
    lang="en",
    model_type="roberta-base",
    verbose=True
)

for i in range(len(candidates)):
    print(f"Example {i+1} BERTScore - P: {P[i]:.4f}, R: {R[i]:.4f}, F1: {F1[i]:.4f}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.92 seconds, 2.16 sentences/sec
Example 1 BERTScore - P: 0.9362, R: 0.9181, F1: 0.9270
Example 2 BERTScore - P: 0.9026, R: 0.8778, F1: 0.8900


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, math

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")

def compute_perplexity(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return math.exp(outputs.loss.item())

for i, cand in enumerate(candidates):
    ppl = compute_perplexity(cand, model, tokenizer)
    print(f"Example {i+1} Perplexity: {ppl:.2f}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Example 1 Perplexity: 65.73
Example 2 Perplexity: 129.86


In [10]:
eval_dataset = [
    {
        "question": "What are the three main components of the attention mechanism?",
        "ground_truth": "The three main components are Query (Q), Key (K), and Value (V) vectors. The attention score is computed as softmax(QK^T/sqrt(d_k))V.",
        "source": "Attention Is All You Need",
    },
    {
        "question": "What is the difference between BERT and GPT in terms of architecture?",
        "ground_truth": "BERT uses a bidirectional Transformer encoder trained with masked language modeling, while GPT uses a unidirectional Transformer decoder trained with autoregressive language modeling.",
        "source": "BERT paper + GPT paper",
    },
    {
        "question": "How does RAG combine retrieval with generation?",
        "ground_truth": "RAG first retrieves relevant documents from a knowledge base using a neural retriever, then conditions the generator on both the input query and retrieved documents to produce the final output.",
        "source": "RAG paper",
    },
    {
        "question": "What is the context length of LLaMA 2?",
        "ground_truth": "LLaMA 2 supports a context length of 4096 tokens, double the 2048 tokens of LLaMA 1.",
        "source": "LLaMA 2 paper",
    },
    {
        "question": "What is in-context learning in GPT-3?",
        "ground_truth": "In-context learning is GPT-3's ability to perform tasks by conditioning on a few examples provided in the prompt, without any gradient updates or fine-tuning.",
        "source": "GPT-3 paper",
    },
]

In [16]:
import pandas as pd
from tqdm import tqdm

results = []
for item in tqdm(eval_dataset):
    rag_result = rag.query(item["question"], method="hybrid", top_k=5, rerank=True)
    results.append({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "generated_answer": rag_result["answer"],
        "retrieved_contexts": [c["text"] for c in rag_result["contexts"]],
        "source": item["source"],
    })

df = pd.DataFrame(results)
print(f"Collected {len(df)} evaluation results")

  0%|          | 0/5 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (536 > 512). Running this sequence through the model will result in indexing errors


Prompt length: 2410 chars, 536 tokens


 20%|██        | 1/5 [00:00<00:02,  1.42it/s]

Prompt length: 2552 chars, 674 tokens


 40%|████      | 2/5 [00:01<00:02,  1.34it/s]

...

Prompt length: 2290 chars, 698 tokens


 80%|████████  | 4/5 [00:02<00:00,  1.58it/s]

Prompt length: 2468 chars, 667 tokens


100%|██████████| 5/5 [00:03<00:00,  1.55it/s]

Collected 5 evaluation results


In [17]:
from bert_score import score as bert_score_fn

# ROUGE
rouge_metric = evaluate.load("rouge")
rouge_results = rouge_metric.compute(
    predictions=df["generated_answer"].tolist(),
    references=df["ground_truth"].tolist()
)
print("Overall ROUGE:", rouge_results)

# BERTScore
P, R, F1 = bert_score_fn(
    df["generated_answer"].tolist(),
    df["ground_truth"].tolist(),
    lang="en", model_type="roberta-base", verbose=True
)
df["bertscore_f1"] = F1.numpy()
print(f"Average BERTScore F1: {F1.mean():.4f}")

Overall ROUGE: {'rouge1': np.float64(0.04253285543608125), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.04253285543608125), 'rougeLsum': np.float64(0.04253285543608125)}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.08 seconds, 62.91 sentences/sec
Average BERTScore F1: 0.7836


In [18]:
print(df)

                                            question  \
0  What are the three main components of the atte...   
1  What is the difference between BERT and GPT in...   
2    How does RAG combine retrieval with generation?   
3             What is the context length of LLaMA 2?   
4              What is in-context learning in GPT-3?   

                                        ground_truth  \
0  The three main components are Query (Q), Key (...   
1  BERT uses a bidirectional Transformer encoder ...   
2  RAG first retrieves relevant documents from a ...   
3  LLaMA 2 supports a context length of 4096 toke...   
4  In-context learning is GPT-3's ability to perf...   

                         generated_answer  \
0  encoder contains self-attention layers   
1                     BERT and OpenAI GPT   
2                                    [iv]   
3                                    [iv]   
4            allow as many demonstrations   

                                  retrieved_contexts  \

In [19]:
def simple_faithfulness_check(answer, contexts):
    """简易版：检查 answer 中的关键名词/数字是否出现在 contexts 中"""
    import re
    answer_tokens = set(re.findall(r'\b[A-Z][a-z]+\b|\b\d+\b', answer))
    context_text = " ".join(contexts)
    if not answer_tokens:
        return 1.0
    supported = sum(1 for t in answer_tokens if t in context_text)
    return supported / len(answer_tokens)

df["faithfulness_approx"] = df.apply(
    lambda row: simple_faithfulness_check(row["generated_answer"], row["retrieved_contexts"]),
    axis=1
)
print(f"Average Faithfulness (approx): {df['faithfulness_approx'].mean():.4f}")

Average Faithfulness (approx): 1.0000


In [20]:
df

,question,ground_truth,generated_answer,retrieved_contexts,source,bertscore_f1,faithfulness_approx
0,What are the three main components of the atte...,"The three main components are Query (Q), Key (...",encoder contains self-attention layers,[Attention mechanisms have become an integral ...,Attention Is All You Need,0.811853,1.0
1,What is the difference between BERT and GPT in...,BERT uses a bidirectional Transformer encoder ...,BERT and OpenAI GPT,"[A.4 Comparison of BERT, ELMo ,and\nOpenAI GPT...",BERT paper + GPT paper,0.841366,1.0
2,How does RAG combine retrieval with generation?,RAG first retrieves relevant documents from a ...,[iv],"[tasks, RAG sets a new state of the art (only ...",RAG paper,0.726843,1.0
3,What is the context length of LLaMA 2?,LLaMA 2 supports a context length of 4096 toke...,[iv],[of 1.0. Figure 5 (a) shows the training loss ...,LLaMA 2 paper,0.735832,1.0
4,What is in-context learning in GPT-3?,In-context learning is GPT-3's ability to perf...,allow as many demonstrations,"[set.\nOn OpenBookQA [MCKS18], GPT-3 improves ...",GPT-3 paper,0.802010,1.0


In [21]:
print("=" * 60)
print("RAG Evaluation Report")
print("=" * 60)
print(f"Total questions: {len(df)}")
print(f"  ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"  ROUGE-L: {rouge_results['rougeL']:.4f}")
print(f"  BERTScore F1: {df['bertscore_f1'].mean():.4f}")
print(f"  Faithfulness (approx): {df['faithfulness_approx'].mean():.4f}")
print("=" * 60)

worst_idx = df["bertscore_f1"].idxmin()
print(f"\nWorst BERTScore example:")
print(f"  Q: {df.loc[worst_idx, 'question']}")
print(f"  Gen: {df.loc[worst_idx, 'generated_answer'][:100]}...")

RAG Evaluation Report
Total questions: 5
  ROUGE-1: 0.0425
  ROUGE-L: 0.0425
  BERTScore F1: 0.7836
  Faithfulness (approx): 1.0000

Worst BERTScore example:
  Q: How does RAG combine retrieval with generation?
  Gen: [iv]...
